In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from scipy.stats import boxcox
import numpy as np
import statsmodels.api as sm

class DataFrameTransformer:
    def __init__(self, df):
        self.df = df

    def factor_to_numeric(self, columns):
        for col in columns:
            self.df[col] = LabelEncoder().fit_transform(self.df[col])
        return self.df

    def convert_dates_to_julian(self, date_column, new_column_name):
        self.df[date_column] = pd.to_datetime(self.df[date_column])
        self.df[new_column_name] = self.df[date_column].apply(lambda x: x.to_julian_date())
        return self.df

    def categorical_to_dummies(self, columns):
        self.df = pd.get_dummies(self.df, columns=columns, drop_first=True)
        return self.df

    def boxcox_transform(self, column):
        self.df[column], _ = boxcox(self.df[column] + 1e-6)
        return self.df

    def tukey_ladder(self, column, power):
        if power == 0:
            self.df[column] = np.log(self.df[column] + 1e-6)
        else:
            self.df[column] = np.sign(self.df[column]) * (np.abs(self.df[column]) ** power)
        return self.df

    def simple_linear_regression(self, y_column, x_column):
        X = sm.add_constant(self.df[x_column])
        y = self.df[y_column]
        model = sm.OLS(y, X).fit()
        return model.summary()

    def multiple_linear_regression(self, y_column, x_columns):
        X = sm.add_constant(self.df[x_columns])
        y = self.df[y_column]
        model = sm.OLS(y, X).fit()
        return model.summary()
